# Dual-Axis GeoFormer â€” train on Colab's free GPU

**Read this before running anything.** The CPU-only prototype's honest, corrected result (see `docs/MANUAL.md` Â§12.3â€“12.4) is:

- Neither GeoFormer nor the U-Net/ResNet-34 baseline detects **buildings or flooding at all** on real SpaceNet-8 imagery -- 0 predictions on both classes, across all 801 real tiles, on both architectures. This was caught by checking *prediction coverage* (how many images a class is ever predicted in), not just an aggregate F1 number -- an earlier version of this project reported "flooded F1 improving" that turned out to be a metric-averaging artifact, not real learning. Coverage is what caught it.
- `road` is the one class either model shows genuine, partial success on (baseline F1 0.43; GeoFormer climbing 0.07â†’0.16â†’0.20 across three checkpoints before the CPU environment's repeated out-of-memory kills stopped that run).

**What this notebook is for**: repeat both training runs at real scale (full 801-tile dataset, a real batch size, as many epochs as you're willing to spend Colab time on) with a GPU, and see whether more compute actually fixes the building/flooded collapse or whether it's a deeper problem (class imbalance, no pretrained backbone, loss function). Read every `evaluate.py` output's **coverage columns**, not just the F1 column, before believing any class has improved.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

In [ ]:
!git clone https://github.com/Redwan002117/D_A_GeoFormer.git
%cd D_A_GeoFormer
!pip install -q boto3 scikit-image timm

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go enable the GPU runtime!)')

## 1. Pull the full real dataset â€” both AOIs, all 801 labeled tiles

The public `spacenet-dataset` S3 bucket needs no AWS credentials (unsigned access). `--aoi all` pulls everything with public labels: Germany (202 tiles) + Louisiana-East (599 tiles) = 801, the complete public-labeled SpaceNet-8 dataset (Louisiana-West exists too but ships imagery only, no public labels -- it's SN-8's own blind competition test set). This is I/O-bound and takes a while the first time; it's resumable (`--append`) and writes `index.json` after every tile, so an interrupted run doesn't lose progress.

In [ ]:
!python prepare_real_data.py --aoi all --n-tiles 599 --out-dir real_sn8_dataset_full

## 2. Train the baseline (U-Net/ResNet-34) â€” the comparison point

No bi-temporal fusion, pre/post images naively channel-stacked -- this is what Table 2 compares GeoFormer against. On the CPU prototype this needed batch size 1 to survive; on a Colab GPU, use a real batch size.

### Optional but recommended: continue from real local progress

The CPU prototype already trained both models to genuine, verified partial progress on this exact dataset before its environment's repeated OOM kills stopped each run (see `docs/MANUAL.md` §12.4): GeoFormer's road F1 climbing past 0.20, baseline's past 0.43. Both checkpoints are real progress, not false starts -- upload them here and resume with a GPU instead of discarding them.

Upload `checkpoints/last.pt` first, then `checkpoints_baseline/last.pt` when prompted again. Skip this cell entirely to train both from scratch instead (also fine -- just slower to reach the same point).

In [ ]:
import os
from google.colab import files
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('checkpoints_baseline', exist_ok=True)

print('Upload your local checkpoints/last.pt (GeoFormer):')
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, 'checkpoints/last.pt')
    print(f'Saved as checkpoints/last.pt ({len(uploaded[fname])} bytes)')

print('Now upload your local checkpoints_baseline/last.pt (baseline):')
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, 'checkpoints_baseline/last.pt')
    print(f'Saved as checkpoints_baseline/last.pt ({len(uploaded[fname])} bytes)')

In [ ]:
import os
resume_flag = '--resume checkpoints_baseline/last.pt' if os.path.exists('checkpoints_baseline/last.pt') else ''
print('Resuming baseline from uploaded checkpoint' if resume_flag else 'Training baseline from scratch')
!python train.py --model baseline --data-dir real_sn8_dataset_full --image-size 256 --batch-size 16 \
                  --epochs 200 --lr 5e-4 --tversky-alpha 0.15 --tversky-beta 0.85 --oversample-rare-classes {resume_flag} \
                  --checkpoint-dir checkpoints_baseline --log-csv training_log_baseline_colab.csv

In [ ]:
!python evaluate.py --checkpoint checkpoints_baseline/best.pt --data-dir real_sn8_dataset_full --image-size 256

**Read the output above like this**: the `Pred imgs` column is the one that matters most. If `building` or `flooded` shows `0` there despite a large `GT imgs` count, that class has collapsed -- the model is not detecting it anywhere, no matter what the F1 number says. Only trust an F1 number for a class whose `Pred imgs` is meaningfully nonzero.

## 3. Train Dual-Axis GeoFormer â€” same data, same comparison

Starting fresh (no `--resume`) is simplest here; pass `--resume checkpoints/last.pt` after uploading the CPU prototype's own checkpoint (see Â§5 below for how it got produced) if you'd rather continue from real, if incomplete, prior training instead of from scratch.

In [ ]:
import os
resume_flag = '--resume checkpoints/last.pt' if os.path.exists('checkpoints/last.pt') else ''
print('Resuming from uploaded checkpoint' if resume_flag else 'Training GeoFormer from scratch')
# efficientnet_b0 is this project's CPU-verified default (docs/MANUAL.md S12.11) --
# on a GPU you can afford a bigger timm backbone, e.g. --pretrained-backbone tf_efficientnet_b3
!python train.py --data-dir real_sn8_dataset_full --image-size 256 --batch-size 16 \
                  --epochs 200 --lr 5e-4 --tversky-alpha 0.15 --tversky-beta 0.85 \
                  --oversample-rare-classes --class-weights "1,2,2,4" \
                  --pretrained-backbone efficientnet_b0 {resume_flag} \
                  --checkpoint-dir checkpoints --log-csv training_log_geoformer_colab.csv

In [ ]:
!python evaluate.py --checkpoint checkpoints/best.pt --data-dir real_sn8_dataset_full --image-size 256

## 4. Plot both, side by side

In [ ]:
!python plot_training_curve.py --log-csv training_log_baseline_colab.csv --out curve_baseline.png --label "baseline, real SpaceNet-8, 801 tiles, GPU"
!python plot_training_curve.py --log-csv training_log_geoformer_colab.csv --out curve_geoformer.png --label "GeoFormer, real SpaceNet-8, 801 tiles, GPU"
from IPython.display import Image, display
display(Image('curve_baseline.png'))
display(Image('curve_geoformer.png'))

## 5. Optional ablations (Table 2's other two rows)

Both are already wired into the CLI -- no code changes needed.

In [ ]:
# Grid-attention ablation: block (local) attention only, global attention removed
!python train.py --data-dir real_sn8_dataset_full --image-size 256 --batch-size 16 \
                  --epochs 60 --lr 1e-3 --tversky-alpha 0.15 --tversky-beta 0.85 --no-grid-attention \
                  --checkpoint-dir checkpoints_no_grid_attn --log-csv training_log_no_grid_attn.csv
!python evaluate.py --checkpoint checkpoints_no_grid_attn/best.pt --data-dir real_sn8_dataset_full --image-size 256

In [ ]:
# Skeleton-bridging ablation (Phase 4) is a demo.py-time flag, not a training-time one --
# it post-processes an already-trained model's road predictions, so it runs against
# whichever GeoFormer checkpoint you already have:
!python demo.py --checkpoint checkpoints/best.pt --skip-bridging

## 6. Download your checkpoints back to your machine

In [ ]:
from google.colab import files
files.download('checkpoints/best.pt')
files.download('checkpoints_baseline/best.pt')

## 7. If building/flooded are STILL collapsed after this

That would be a real, informative result -- it would mean the problem isn't data volume or compute (both are now much larger than the CPU prototype had), and points toward the architecture/training-recipe causes `docs/MANUAL.md` Â§13 already names: no pretrained backbone (this repo's MaxViT encoder is randomly initialized, not ImageNet-21k-pretrained like the thesis's Phase 2 calls for), no class-balanced sampling or focal-loss-style handling of the severe imbalance (`flooded` is under 1% of pixels), and Tversky's alpha/beta alone may not be enough leverage against that imbalance. Try, in roughly this order of effort:

1. A `timm` MaxViT-Base backbone (ImageNet-21k pretrained) in place of `model.py`'s from-scratch encoder.
2. A weighted/oversampled `DataLoader` so `flooded`-containing tiles appear more often per epoch than their raw ~25% prevalence.
3. More epochs than 60 -- `road`'s own improvement (0.07â†’0.20 in the CPU run) was still climbing, not plateaued, when that run was cut short.